# LAID gate 2: Community Forensics multi-generator benchmark

This evaluates the same baseline on seven generator families using Unbiased Tiny GenImage. It selects 100 AI images per generator and 100 distinct real images for each comparison, then repeats the fixed `0.65` evaluation after JPEG and downscale stress.

Before running: enable GPU and Internet, click **Add Input**, and attach `cartografia/unbiased-tiny-genimage` (about 2.5 GB). This is evaluation only; no training occurs.

In [ ]:
import shutil
import subprocess
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU attached. Enable a GPU in Session options and reconnect.')
inputs = Path('/kaggle/input')
required = ['Nature', 'ADM', 'BigGAN', 'glide', 'Midjourney', 'stable_diffusion_v_1_5', 'VQDM', 'wukong']
missing = [name for name in required if len(list(inputs.rglob(name))) != 1]
if missing:
    raise RuntimeError(f'Unbiased Tiny GenImage is missing folders: {missing}. Attach cartografia/unbiased-tiny-genimage.')
disk = shutil.disk_usage('/kaggle/working')
if disk.free < 10 * 1024**3:
    raise RuntimeError(f'Less than 10 GB working disk free: {disk.free / 1024**3:.1f} GB')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Free working disk: {disk.free / 1024**3:.1f} GB')


In [ ]:
laid = Path('/kaggle/working/laid')
if (laid / '.git').exists():
    subprocess.run(['git', '-C', str(laid), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/Eienel/laid.git', str(laid)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(laid), 'timm==1.0.15', 'huggingface-hub'], check=True)

official = Path('/kaggle/working/community-forensics')
commit = 'ee5b71d43db0f3779e1edd64ee927b13f2dd6ad4'
if not (official / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/JeongsooP/Community-Forensics.git', str(official)], check=True)
subprocess.run(['git', '-C', str(official), 'checkout', '--detach', commit], check=True)


In [ ]:
results = laid / 'results' / 'community-forensics-224-multigen'
subprocess.run([
    'python', str(laid / 'scripts' / 'kaggle_commfor_baseline.py'),
    '--input-root', str(inputs), '--dataset-kind', 'multigen',
    '--per-class', '100', '--batch-size', '32', '--output-dir', str(results)
], check=True)


In [ ]:
import json
report = json.loads((results / 'report.json').read_text())
provenance = json.loads((results / 'provenance.json').read_text())
print('Overall balanced accuracy:', f"{report['overall']['balanced_accuracy']:.3f}")
print('By generator:')
for name, values in report['by_generator'].items():
    print(' ', name, f"{values['balanced_accuracy']:.3f}")
print('By degradation:')
for name, values in report['by_degradation'].items():
    print(' ', name, f"{values['balanced_accuracy']:.3f}")
print('Mean GPU ms/image:', provenance['mean_inference_ms_per_image'])
archive = shutil.make_archive('/kaggle/working/laid-multigen-results', 'zip', results)
print('Result archive:', archive)


## Finish

Send a screenshot of the final cell. Refresh `/kaggle/working` in the Output panel and download `laid-multigen-results.zip`, then stop the GPU session.